# Does Visualization Style Shift Attention Away from Pseudocode?
## An Eye-Tracking Analysis of Algorithm Animation Design

---

**Dataset:** Fathi Fall 2022 — Tobii Pro eye-tracking, algorithm visualization study  
**Prepared:** April 2026

---

## Abstract

Algorithm visualizations typically present two parallel representations: a step-by-step pseudocode listing on one side and a dynamic visual output (e.g., a geospatial graph) on the other. How viewers allocate attention between these two representations — and whether that allocation is shaped by the visualization's graphical style — remains underexplored. Using eye-tracking data collected across three visualization styles (IRN, Galles, Metal) and two algorithm types (BFS, DFS), we compute a *pseudocode/map attention ratio* for each participant. We then ask: (1) Does this ratio differ between Galles and Metal designs? (2) Does it differ between BFS and DFS within each design? (3) Does a high pseudocode-to-map attention ratio predict increased fixation on the Queue AOI — the data structure that mechanistically connects the two panels? Results are interpreted in terms of cognitive integration during algorithm comprehension.

---

## 1. Introduction

When a learner watches a Breadth-First Search animation, they face a dual-channel representation: on the left, pseudocode lines execute sequentially; on the right, nodes on a spatial graph are visited in a spreading wave. These two representations are informationally redundant — both describe the same algorithm — but they are cognitively distinct. Pseudocode requires symbolic, step-level processing. The graph demands spatial, gestalt processing.

Prior work on split-attention effects (Sweller, 1994; Mayer, 2001) suggests that forcing learners to mentally integrate two spatially separated representations imposes extraneous cognitive load. If a visualization's design makes the graphical output more visually compelling — brighter colors, richer node styling, more dramatic animation — viewers may preferentially attend to the map and underprocess the pseudocode. This would be a design failure for any instructor who intends the pseudocode to be read.

The present analysis focuses on a single, narrow question:

> **Does the ratio of total fixation duration on the Pseudocode AOI to total fixation duration on the Geospatial Map AOI differ between the Galles and Metal visualization styles in BFS, and does this ratio predict the number of fixations on the Queue AOI?**

The Queue AOI is the intermediate data structure — a first-in, first-out list of nodes yet to be visited. A viewer who reads the pseudocode carefully should also look at the queue to track the algorithm's state. If the pseudocode/map ratio positively predicts queue fixation count, it suggests that pseudocode-dominant viewers are processing the algorithm mechanistically (code → queue → map) rather than skimming visually.

### 1.1 Study Design

Participants were divided into three groups and shown algorithm animations in one of three visualization styles:
- **IRN** — a geospatially realistic node layout
- **Galles** — a structured, textbook-style node layout
- **Metal** — a high-contrast, stylized rendering

Each participant watched both BFS and DFS animations. Eye movements were recorded with a Tobii eye-tracker at 60 Hz. Areas of Interest (AOIs) were defined for: Pseudocode (left panel), Geospatial Map (right panel), Queue, and Stack.

Aggregate fixation metrics (total fixation duration, fixation count, visit count, time-to-first-fixation) were exported per participant per AOI using Tobii Studio.

---

## 2. Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')
print('Libraries loaded.')

### 2.1 Loading Aggregate XLS Files

Each `.xls` file contains one row per participant and one set of columns per AOI metric. The XLS files cover Galles and Metal visualizations across Groups 1–3. IRN data is encoded in the raw TSV export and processed separately.

In [ ]:
file_map = {
    'Group1_gallesBFSData.xls': ('Group1', 'Galles', 'BFS'),
    'Group1_gallesDFSData.xls': ('Group1', 'Galles', 'DFS'),
    'Group1_metalBFSData.xls':  ('Group1', 'Metal',  'BFS'),
    'Group1_metalDFSData.xls':  ('Group1', 'Metal',  'DFS'),
    'Group2_gallesBFSData.xls': ('Group2', 'Galles', 'BFS'),
    'Group2_gallesDFSData.xls': ('Group2', 'Galles', 'DFS'),
    'Group2_metalBFSData.xls':  ('Group2', 'Metal',  'BFS'),
    'Group2_metalDFSData.xls':  ('Group2', 'Metal',  'DFS'),
    'Group3_gallesBFSData.xls': ('Group3', 'Galles', 'BFS'),
    'Group3_gallesDFSData.xls': ('Group3', 'Galles', 'DFS'),
    'Group3_metalBFSData.xls':  ('Group3', 'Metal',  'BFS'),
    'Group3_metalDFSData.xls':  ('Group3', 'Metal',  'DFS'),
}

def extract_aoi_mean(df, metric, aoi_keyword):
    """Return the _Mean column for a given metric string and AOI keyword."""
    candidates = [
        c for c in df.columns
        if metric in c
        and aoi_keyword.lower() in c.lower()
        and c.endswith('_Mean')
        and 'Include Zeros' not in c
    ]
    return candidates[0] if candidates else None

records = []
for fname, (group, vis, algo) in file_map.items():
    path = DATA_DIR / fname
    if not path.exists():
        print(f'  MISSING: {fname}')
        continue
    df = pd.read_excel(path, engine='xlrd')
    df = df.rename(columns={df.columns[0]: 'participant'})

    tfd_pseudo = extract_aoi_mean(df, 'Total Fixation Duration', 'rectangle')
    tfd_map    = extract_aoi_mean(df, 'Total Fixation Duration', 'rectangle 2')
    fc_pseudo  = extract_aoi_mean(df, 'Fixation Count', 'rectangle')
    fc_map     = extract_aoi_mean(df, 'Fixation Count', 'rectangle 2')
    ttff       = extract_aoi_mean(df, 'Time to First Fixation', 'rectangle')

    for _, row in df.iterrows():
        pname = str(row.get('participant', '')).strip()
        if not pname or pname.lower() in ('nan', 'mean', 'sum', 'std'):
            continue
        records.append({
            'participant':    pname,
            'group':          group,
            'vis_style':      vis,
            'algorithm':      algo,
            'tfd_pseudocode': pd.to_numeric(row.get(tfd_pseudo), errors='coerce'),
            'tfd_map':        pd.to_numeric(row.get(tfd_map),    errors='coerce'),
            'fc_pseudocode':  pd.to_numeric(row.get(fc_pseudo),  errors='coerce'),
            'fc_map':         pd.to_numeric(row.get(fc_map),     errors='coerce'),
            'ttff_pseudocode':pd.to_numeric(row.get(ttff),       errors='coerce'),
        })

df_main = pd.DataFrame(records)
df_main['pseudo_map_ratio'] = df_main['tfd_pseudocode'] / (df_main['tfd_map'] + 1e-9)

print(f'Participants loaded: {len(df_main)}')
print(f'Vis styles:          {df_main["vis_style"].unique()}')
print(f'Algorithms:          {df_main["algorithm"].unique()}')
df_main[['participant', 'vis_style', 'algorithm', 'tfd_pseudocode', 'tfd_map', 'pseudo_map_ratio']].head(8)

### 2.2 Loading Raw Gaze Data (IRN Visualization)

IRN fixation data is extracted from the full Tobii export (`Fathi Fall 22_Data_Export (3).tsv`). Each row in the TSV is one gaze sample; fixation events are identified by `GazeEventType == 'Fixation'` and collapsed per `FixationIndex`.

In [ ]:
tsv_path = DATA_DIR / 'Fathi Fall 22_Data_Export (3).tsv'
irn_raw = pd.read_csv(tsv_path, sep='\t', encoding='utf-8-sig', low_memory=False)
irn_fix = irn_raw[irn_raw['GazeEventType'].astype(str).str.strip().str.lower() == 'fixation'].copy()

# Identify AOI columns
aoi_cols = [c for c in irn_fix.columns if 'AOI' in c]
pseudo_col = next((c for c in aoi_cols if 'Pseudocode' in c), None)
queue_col  = next((c for c in aoi_cols if 'Queue' in c or 'queue' in c), None)
map_col    = next((c for c in aoi_cols if 'Map' in c or 'Geospatial' in c), None)
stack_col  = next((c for c in aoi_cols if 'stack' in c.lower()), None)

print(f'Total fixation samples: {len(irn_fix):,}')
print(f'Participants: {irn_fix["ParticipantName"].nunique()}')
print(f'AOI columns found — Pseudocode: {pseudo_col} | Queue: {queue_col} | Map: {map_col} | Stack: {stack_col}')

In [ ]:
# Collapse to per-fixation-event rows
def on_aoi(group, col):
    if col is None or col not in group.columns:
        return 0
    return int((pd.to_numeric(group[col], errors='coerce').fillna(-1) == 1).any())

fix_events = (
    irn_fix
    .groupby(['ParticipantName', 'FixationIndex'])
    .apply(lambda g: pd.Series({
        'duration_ms':   float(g['GazeEventDuration'].iloc[0]),
        'on_pseudocode': on_aoi(g, pseudo_col),
        'on_queue':      on_aoi(g, queue_col),
        'on_map':        on_aoi(g, map_col),
        'on_stack':      on_aoi(g, stack_col),
    }))
    .reset_index()
)

# Per-participant summary
def tfd_on(x, mask):
    return x[mask].sum() / 1000  # ms → s

irn_ppt = (
    fix_events
    .groupby('ParticipantName')
    .apply(lambda g: pd.Series({
        'tfd_pseudocode': tfd_on(g['duration_ms'], g['on_pseudocode'] == 1),
        'tfd_map':        tfd_on(g['duration_ms'], g['on_map'] == 1),
        'fc_pseudocode':  int(g['on_pseudocode'].sum()),
        'fc_queue':       int(g['on_queue'].sum()),
        'fc_map':         int(g['on_map'].sum()),
        'fc_stack':       int(g['on_stack'].sum()),
        'total_fixations': g['FixationIndex'].nunique(),
    }))
    .reset_index()
)
irn_ppt['pseudo_map_ratio'] = irn_ppt['tfd_pseudocode'] / (irn_ppt['tfd_map'] + 1e-9)
irn_ppt['vis_style'] = 'IRN'

print(irn_ppt.to_string())

---

## 3. Descriptive Statistics

Before testing hypotheses, we examine the distributions of AOI fixation metrics across visualization styles and algorithms. This surfaces any floor/ceiling effects, extreme outliers, or strong main effects that might constrain interpretation.

In [ ]:
desc = (
    df_main
    .groupby(['vis_style', 'algorithm'])[['tfd_pseudocode', 'tfd_map', 'fc_pseudocode', 'fc_map', 'pseudo_map_ratio']]
    .agg(['mean', 'median', 'std', 'count'])
)
desc.columns = ['_'.join(c) for c in desc.columns]
print('=== Descriptive Statistics by Visualization Style × Algorithm ===')
print(desc.round(3).to_string())

In [ ]:
# Figure 1 — Distribution of core fixation metrics
palette = {'Galles': '#4C72B0', 'Metal': '#DD8452'}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(
    'Figure 1. AOI Fixation Metrics by Visualization Style and Algorithm',
    fontsize=13, fontweight='bold', y=1.01
)

metrics = [
    ('tfd_pseudocode', 'Total Fixation Duration — Pseudocode (s)'),
    ('tfd_map',        'Total Fixation Duration — Geospatial Map (s)'),
    ('fc_pseudocode',  'Fixation Count — Pseudocode'),
    ('fc_map',         'Fixation Count — Geospatial Map'),
]

for ax, (metric, label) in zip(axes.flat, metrics):
    data = df_main[df_main[metric].notna()]
    sns.boxplot(
        data=data, x='algorithm', y=metric, hue='vis_style',
        palette=palette, ax=ax, width=0.5, linewidth=1.2
    )
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Algorithm', fontsize=10)
    ax.set_ylabel('')
    ax.legend(title='Style', fontsize=9)

plt.tight_layout()
plt.savefig('fig1_aoi_distributions.png', bbox_inches='tight')
plt.show()

---

## 4. Primary Analysis: Pseudocode/Map Attention Ratio

We operationalize attention allocation as:

$$
\text{Ratio} = \frac{\text{TFD}_{\text{pseudocode}}}{\text{TFD}_{\text{map}}}
$$

where TFD = Total Fixation Duration (seconds). A ratio > 1 indicates that the participant spent more cumulative fixation time on the pseudocode than the map. A ratio < 1 indicates the reverse.

Because sample sizes are small (n ≈ 7–10 per cell) and normality cannot be assumed, we use the Mann-Whitney U test for between-group comparisons. Effect size is reported as rank-biserial correlation *r*.

### 4.1 Does Visualization Style Affect the Ratio? (BFS)

In [ ]:
bfs = df_main[df_main['algorithm'] == 'BFS'].copy()
dfs = df_main[df_main['algorithm'] == 'DFS'].copy()

galles_bfs = bfs[bfs['vis_style'] == 'Galles']['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan).dropna()
metal_bfs  = bfs[bfs['vis_style'] == 'Metal']['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan).dropna()

u, p = stats.mannwhitneyu(galles_bfs, metal_bfs, alternative='two-sided')
r_rb = 1 - (2 * u) / (len(galles_bfs) * len(metal_bfs))

print('=== BFS: Galles vs Metal (Pseudocode/Map Ratio) ===')
print(f'  Galles  n={len(galles_bfs)}, median={galles_bfs.median():.3f}, mean={galles_bfs.mean():.3f}')
print(f'  Metal   n={len(metal_bfs)},  median={metal_bfs.median():.3f}, mean={metal_bfs.mean():.3f}')
print(f'  Mann-Whitney U = {u:.1f}, p = {p:.4f}')
print(f'  Rank-biserial r = {r_rb:.3f}  (medium ≥ 0.3, large ≥ 0.5)')

In [ ]:
# Figure 2 — Ratio by vis style × algorithm
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'Figure 2. Pseudocode/Map Attention Ratio by Visualization Style',
    fontsize=13, fontweight='bold'
)

for ax, (algo, data) in zip(axes, [('BFS', bfs), ('DFS', dfs)]):
    plot = data[data['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan).notna()].copy()
    plot = plot[plot['pseudo_map_ratio'] < 30]  # clip extreme outliers for display
    
    sns.stripplot(
        data=plot, x='vis_style', y='pseudo_map_ratio',
        palette=palette, jitter=True, alpha=0.75, size=7,
        order=['Galles', 'Metal'], ax=ax
    )
    sns.boxplot(
        data=plot, x='vis_style', y='pseudo_map_ratio',
        palette=palette, ax=ax, width=0.35, linewidth=1.5,
        boxprops=dict(alpha=0.35), showfliers=False,
        order=['Galles', 'Metal']
    )
    ax.axhline(1.0, color='#555', linestyle='--', linewidth=1.1, label='Equal attention (ratio = 1)')
    ax.set_title(f'{algo} Algorithm', fontsize=12)
    ax.set_xlabel('Visualization Style', fontsize=11)
    ax.set_ylabel('Pseudocode TFD / Map TFD', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig2_ratio_by_style.png', bbox_inches='tight')
plt.show()

### 4.2 Does Algorithm Type (BFS vs DFS) Affect the Ratio Within Each Style?

In [ ]:
print('=== BFS vs DFS: Ratio by Visualization Style ===')
for vis in ['Galles', 'Metal']:
    sub = df_main[df_main['vis_style'] == vis]
    bfs_r = sub[sub['algorithm'] == 'BFS']['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan).dropna()
    dfs_r = sub[sub['algorithm'] == 'DFS']['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(bfs_r) > 1 and len(dfs_r) > 1:
        u, p = stats.mannwhitneyu(bfs_r, dfs_r, alternative='two-sided')
        r = 1 - (2 * u) / (len(bfs_r) * len(dfs_r))
        print(f'\n  {vis}')
        print(f'    BFS  n={len(bfs_r)}, median={bfs_r.median():.3f}')
        print(f'    DFS  n={len(dfs_r)}, median={dfs_r.median():.3f}')
        print(f'    U={u:.0f}, p={p:.4f}, r={r:.3f}')
    else:
        print(f'  {vis}: insufficient data for test')

---

## 5. Does the Ratio Predict Queue Fixation Count? (IRN Data)

The Queue AOI represents the frontier of the BFS traversal — a list of nodes waiting to be visited. If a viewer is processing the algorithm procedurally (following the pseudocode), we expect them to also monitor the queue to confirm the current state. The prediction is:

> **Participants with higher pseudocode/map ratios will have more fixations on the Queue AOI** (Spearman correlation).

This analysis uses the IRN TSV export, which includes Queue AOI columns not available in the XLS aggregates.

In [ ]:
x = irn_ppt['pseudo_map_ratio'].replace([np.inf, -np.inf], np.nan)
y = irn_ppt['fc_queue']
valid = x.notna() & y.notna()
x_v, y_v = x[valid], y[valid]

print(f'Valid participants for correlation: n={len(x_v)}')

if len(x_v) >= 3:
    r_sp, p_sp = stats.spearmanr(x_v, y_v)
    r_pe, p_pe = stats.pearsonr(x_v, y_v)
    print(f'Spearman r = {r_sp:.3f},  p = {p_sp:.4f}')
    print(f'Pearson  r = {r_pe:.3f},  p = {p_pe:.4f}')
else:
    print('Insufficient participants for reliable correlation (n < 3).')
    print('Note: the IRN TSV export may contain only one participant. See limitations section.')

In [ ]:
if len(x_v) >= 3:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(x_v, y_v, color='#4C72B0', s=80, alpha=0.85, edgecolors='white', linewidth=0.5, zorder=3)
    m, b = np.polyfit(x_v, y_v, 1)
    xline = np.linspace(x_v.min(), x_v.max(), 100)
    ax.plot(xline, m * xline + b, color='tomato', linewidth=1.8, linestyle='--',
            label=f'Spearman r = {r_sp:.2f}, p = {p_sp:.3f}')
    ax.set_xlabel('Pseudocode / Map Attention Ratio', fontsize=11)
    ax.set_ylabel('Queue Fixation Count', fontsize=11)
    ax.set_title('Figure 3. Does Pseudocode Attention Predict Queue Engagement?\n(IRN visualization, BFS)', fontsize=11)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('fig3_ratio_vs_queue.png', bbox_inches='tight')
    plt.show()
else:
    print('Figure 3 skipped (insufficient participants in IRN TSV export).')

---

## 6. Correlation Matrix

As an exploratory supplement, we examine the full correlation structure among AOI fixation metrics collapsed across all visualization styles. This surfaces any unexpected relationships and identifies which metrics co-vary most strongly.

In [ ]:
numeric_cols = ['tfd_pseudocode', 'tfd_map', 'fc_pseudocode', 'fc_map', 'pseudo_map_ratio', 'ttff_pseudocode']
corr_data = df_main[numeric_cols].replace([np.inf, -np.inf], np.nan).dropna()
corr = corr_data.corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.75}
)
ax.set_title('Figure 4. Spearman Correlation Matrix — AOI Fixation Metrics\n(all participants, all conditions)', fontsize=11)
plt.tight_layout()
plt.savefig('fig4_correlation_matrix.png', bbox_inches='tight')
plt.show()

---

## 7. Summary Figure

In [ ]:
plot_data = (
    df_main
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['pseudo_map_ratio'])
    .query('pseudo_map_ratio < 25')
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(
    data=plot_data, x='vis_style', y='pseudo_map_ratio', hue='algorithm',
    palette={'BFS': '#4C72B0', 'DFS': '#DD8452'},
    split=True, inner='quartile', ax=ax, linewidth=1, order=['Galles', 'Metal']
)
ax.axhline(1.0, color='#333', linestyle='--', linewidth=1.1, alpha=0.6, label='Equal attention (ratio = 1)')
ax.set_title(
    'Figure 5. Pseudocode/Map Attention Ratio: Visualization Style × Algorithm\n'
    '(split violin; inner lines = quartiles)',
    fontsize=11
)
ax.set_xlabel('Visualization Style', fontsize=11)
ax.set_ylabel('Pseudocode TFD / Map TFD', fontsize=11)
ax.legend(title='Algorithm', fontsize=10)
plt.tight_layout()
plt.savefig('fig5_summary_violin.png', bbox_inches='tight')
plt.show()

---

## 8. Discussion

### 8.1 Visualization Style and Attention Allocation

The central question was whether Galles and Metal visualizations elicit different pseudocode/map attention ratios. A significant difference (small or medium effect) would suggest that the graphical design of the animation — not just the algorithm being shown — shapes whether viewers read the code or watch the graph.

If Galles produces a higher ratio than Metal: the Metal visualization's richer or more dynamic graphical output may draw the eye away from the pseudocode, a split-attention effect consistent with Sweller's cognitive load theory. Designers relying on Metal-style visualizations for instruction may need to either spatially integrate the pseudocode with the graph or use progressive reveal techniques to force code engagement.

If the two styles do not significantly differ: attention allocation may be driven primarily by individual differences in learning strategy (pseudocode-reader vs. graph-watcher) or by algorithm structure rather than visual design.

### 8.2 Algorithm Complexity and Ratio

If BFS consistently produces a higher pseudocode/map ratio than DFS: this could reflect the fact that BFS's level-order traversal is harder to track visually on a spatial graph (the "ring" expands in all directions simultaneously), so viewers look at the pseudocode for orientation. DFS's depth-first path is visually traceable as a single line descending the graph, requiring less code consultation. This would be a pedagogically important finding: visualizations of BFS may need to put more scaffolding on the pseudocode panel precisely when viewers are already finding it harder to follow.

### 8.3 Queue Fixations as a Behavioral Marker

The predicted positive correlation between the pseudocode/map ratio and queue fixation count provides a mechanism test. If confirmed, it supports the interpretation that pseudocode-dominant viewers are performing a three-step integration: (1) read the pseudocode step, (2) check the queue to verify the algorithm's state, (3) confirm the graph output. This is a qualitatively different cognitive strategy than going directly from the pseudocode to the graph — and a more robust one for actual learning.

### 8.4 Limitations

1. **AOI naming ambiguity.** The XLS files label the two AOIs only as 'Rectangle' and 'Rectangle 2'. The assignment of Rectangle = Pseudocode and Rectangle 2 = Map was inferred from the spatial layout described in the AOI `.txt` files and should be verified against the original Tobii Studio project before drawing conclusions.

2. **Small cell sizes.** With approximately 7–10 participants per visualization style per algorithm, statistical power is low. Non-significant results cannot be interpreted as evidence of no effect; only large effects are reliably detectable.

3. **No comprehension outcome.** Fixation patterns are correlates of processing, not direct measures of learning. The analysis cannot determine whether pseudocode-dominant attention leads to better or worse comprehension without an accompanying knowledge test.

4. **Single participant in IRN TSV.** If the raw TSV export contains only one participant, the queue fixation correlation analysis collapses. Future analysis should locate a multi-participant IRN export.

5. **No trial ordering control.** If some participants watched BFS before DFS (or vice versa), practice effects could confound algorithm comparisons.

---

## 9. References

- Mayer, R. E. (2001). *Multimedia learning*. Cambridge University Press.
- Sweller, J. (1994). Cognitive load theory, learning difficulty, and instructional design. *Learning and Instruction, 4*(4), 295–312.
- Tobii Pro (2023). *Tobii Studio user manual*. Tobii AB.
- Price, B. A., Baecker, R. M., & Small, I. S. (1993). A principled taxonomy of software visualization. *Journal of Visual Languages and Computing, 4*(3), 211–266.